In [14]:
#不同算法的全局带宽总和对比
import pandas as pd

# 设置pandas选项，确保输出所有行和所有列，不折叠省略
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df = pd.read_csv('./Data/Evaluation/H100_Real/H100_26H100_27H100_28H100_29/multi_tenant_simulation.csv')
# 按需分组提取并展示全部内容
grouped = df.loc[df.groupby(['repeat_id','algorithm_name', 'search_mode'])['job_id'].idxmax()]
# grouped = grouped[grouped['algorithm_name'] != 'GroundTruth']

# 先输出原有字段，再新增 best_throughput 列

# 找到每个repeat_id下real_cluster_throughput最大值的那些algorithm_name
max_throughput = grouped.groupby('repeat_id')['real_cluster_throughput'].max()
# 找到每个repeat_id下达到最大throughput的行
max_rows = grouped[grouped.apply(lambda row: row['real_cluster_throughput'] == max_throughput[row['repeat_id']], axis=1)]
# 统计每个算法出现的次数
algo_counts = max_rows['algorithm_name'].value_counts()
print("各 repeat_id 下 real_cluster_throughput 达到最大值的算法及出现次数：")
for algo, count in algo_counts.items():
    print(f"  {algo}: {count} 次")
# 打印每个算法在哪些repeat_id下达到最大值
print("\n各算法达到最大值的 repeat_id：")
for algo in algo_counts.index:
    repeat_ids = max_rows[max_rows['algorithm_name'] == algo]['repeat_id'].tolist()
    print(f"  {algo}: {repeat_ids}")


各 repeat_id 下 real_cluster_throughput 达到最大值的算法及出现次数：
  GroundTruth: 86 次
  UpperBandDisp: 55 次
  BandDisp: 44 次
  Topo: 36 次
  Default: 21 次

各算法达到最大值的 repeat_id：
  GroundTruth: [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 51, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 79, 82, 83, 84, 85, 86, 87, 88, 89, 91, 92, 93, 94, 96, 97, 99]
  UpperBandDisp: [1, 4, 9, 10, 12, 14, 15, 16, 17, 19, 22, 23, 24, 28, 29, 31, 32, 33, 34, 35, 38, 39, 41, 43, 45, 46, 47, 52, 53, 54, 55, 57, 60, 61, 62, 63, 64, 65, 67, 74, 76, 77, 78, 79, 83, 84, 86, 88, 89, 91, 94, 95, 97, 98, 99]
  BandDisp: [1, 4, 9, 10, 12, 16, 17, 19, 23, 26, 28, 29, 32, 34, 35, 37, 41, 42, 43, 46, 47, 52, 53, 54, 56, 57, 61, 63, 64, 65, 71, 76, 78, 79, 82, 83, 84, 86, 88, 91, 94, 97, 98, 99]
  Topo: [1, 2, 5, 7, 11, 12, 15, 19, 27, 30, 31, 33, 34, 35, 37,

In [15]:
result = grouped[['repeat_id','job_id', 'algorithm_name', 'search_mode',  'real_cluster_throughput']].copy()
# 计算每个repeat_id下最大的real_cluster_throughput，并赋值给新列
best_throughput_map = grouped.groupby('repeat_id')['real_cluster_throughput'].max()
result['best_throughput'] = result['repeat_id'].map(best_throughput_map)
result["throughput_ratio"] = (result["real_cluster_throughput"] / result["best_throughput"]).round(2)

# 计算每个repeat_id下不同算法的real_final_bw的标准差
# 首先需要从原始df中获取每个repeat_id和algorithm_name对应的所有real_final_bw
bw_std_map = df.groupby(['repeat_id', 'algorithm_name'])['real_final_bw'].std().round(2).reset_index()
bw_std_map.columns = ['repeat_id', 'algorithm_name', 'real_final_bw_std']
# 将标准差合并到result中
result = result.merge(bw_std_map, on=['repeat_id', 'algorithm_name'], how='left')

# 筛选 repeat_id <= x 的数据
x = 100
filtered_result = result[result['repeat_id'] <= x]

# 按 repeat_id 分组显示，每组之间添加分割线
# for repeat_id in sorted(filtered_result['repeat_id'].unique()):
#     print(f"\n{'='*80}")
#     print(f"Repeat ID: {repeat_id}")
#     print(f"{'='*80}")
#     group_data = filtered_result[filtered_result['repeat_id'] == repeat_id]
#     print(group_data.to_string(index=False))

# 计算每个算法在repeat_id <= x 下的throughput_ratio总和
print(f'算法在repeat = {x} 下的throughput_ratio总和')
print(filtered_result.groupby('algorithm_name')['throughput_ratio'].sum().sort_values(ascending=False))

print(f'算法在repeat = {x} 下,多次workload分配的全局带宽的标准差的均值')
print(filtered_result.groupby('algorithm_name')['real_final_bw_std'].mean().sort_values(ascending=True))

算法在repeat = 100 下的throughput_ratio总和
algorithm_name
GroundTruth      97.54
UpperBandDisp    93.08
BandDisp         87.71
Topo             85.81
Default          80.79
Random           13.85
Name: throughput_ratio, dtype: float64
算法在repeat = 100 下,多次workload分配的全局带宽的标准差的均值
algorithm_name
Random           34.2918
GroundTruth      60.8367
UpperBandDisp    65.4121
BandDisp         68.7626
Topo             72.0246
Default          76.6195
Name: real_final_bw_std, dtype: float64


In [48]:
df = pd.read_csv('./Data/Evaluation/H100_Real/H100_26H100_27H100_28H100_29/multi_tenant_simulation.csv')
result = df[(df['repeat_id'] == 9) & (df['algorithm_name'].isin(['GroundTruth','BandDisp','UpperBandDisp', 'Topo']))]
result = result.drop(columns=['search_mode'])
result.iloc[:, 1:]

FileNotFoundError: [Errno 2] No such file or directory: './Data/Evaluation/H100_Real/H100_26H100_27H100_28H100_29/multi_tenant_simulation.csv'